In [2]:
class BlogPlatform:
   def __init__(self, db_path='blog_data.db', port=8000):
        """プラットフォームを初期化
        
        各技術コンポーネントをインスタンス化して連携させる。
        """
        # --- 技術6: データベース ---
        self.db = Database(db_path)
        
        # --- キャッシュ (DB最適化) ---
        self.cache_server = CacheServer(host='127.0.0.1', port=6379)
        self.cache_server.start()
        self.cache = CacheClient(self.cache_server)
        
        # --- 技術4: Webサーバー ---
        self.web = WebServer(host='127.0.0.1', port=port)
        
        # --- 技術5: Webブラウザ (テスト・プレビュー用) ---
        self.browser = Browser(base_url=f'http://127.0.0.1:{port}')
        
        # --- 技術1: テキストエディタ (記事編集用) ---
        self.editors = {}  # post_id -> TextEditor のマッピング
        
        # 初期データの投入
        self._initialize_db()
        
        # ルーティングの設定
        self._setup_routes()
    

In [ ]:
   def _initialize_db(self):
        """データベースにサンプルデータを投入（技術6: Database）"""
        try:
            self.db.get('post_counter')   #重複を防ぐ、post_counter とは
                                            #記事に番号をつけるためのカウンター です
        except KeyError:
            self.db.set('post_counter', 0)
            
            sample_posts = [
                {
                    'title': 'Pythonでデータベースを自作する',
                    'author': '山田太郎',
                    'content': ('イミュータブルなバイナリツリーを使って、'
                                'キー/バリューストアを実装する方法を解説します。\n\n'
                                'データ構造の不変性を保つことで、並行処理での安全性が高まります。'
                                'また、コピーオンライトにより効率的なメモリ使用が可能です。'),
                    'tags': ['Python', 'Database', 'Tutorial'],
                    'date': '2024-01-01'
                },
              
            ]
            
            for post in sample_posts:
                self._create_post(post) #自分の _create_post 関数に、今取り出した post を渡して実行する
             
             ##

Self.〜〜で呼び出し関数
Selfに蓄積

In [ ]:
def _setup_routes(self):    #「URLとその処理担当者を紐づける受付案内板を作る
        """URLルーティングの設定（技術4: デコレータベースルーティング）"""
        
        @self.web.route('/')
        def index(request, query):
            return self._render_index() 
        #request = リクエスト情報, query = URLの?以降」その後に続くお客様情報よこせ
        
        @self.web.route('/posts')
        def list_posts(request, query):
            return self._render_post_list() #request = リクエスト情報, query = URLの?以降」
        
        @self.web.route('/post/<id>')
        def view_post(request, query):
            post_id = request.path.split('/')[-1]
            ## 「リクエストのURLパスを / で分割して最後の要素を取り出す」
            return self._render_post(post_id)  
        ## 例: '/post/post_0' → ['','post','post_0'] → 'post_0' ID探し
        
        @self.web.route('/search')
        def search(request, query):
            search_query = query.get('q', [''])[0]
            return self._render_search_results(search_query)
        
        @self.web.route('/editor')
        def new_editor(request, query):
            """新規記事エディタ（技術1: テキストエディタ）"""
            return self._render_editor_page()
        
        @self.web.route('/editor/<id>')
        def edit_post(request, query):
            """記事編集エディタ（技術1: テキストエディタ）"""
            post_id = request.path.split('/')[-1]
            #['', 'editor', 'post_0']分割結果の１番最後とりだし
            return self._render_editor_page(post_id)
        
        @self.web.route('/preview/<id>')
        def preview_post(request, query):
            """記事プレビュー（技術5: ブラウザのHTMLパース）"""
            post_id = request.path.split('/')[-1]
            return self._render_preview(post_id)
        
        # --- APIルート ---
        
        @self.web.route('/api/posts', method='GET')
        def api_list_posts(request, query):
            return self._get_all_posts()
        
        @self.web.route('/api/post', method='POST')
        def api_create_post(request, data):
            return self._create_post(data)
        
        @self.web.route('/api/post/<id>', method='GET')
        def api_get_post(request, query):
            post_id = request.path.split('/')[-1]
            return self._get_post(post_id)
        
        @self.web.route('/api/search', method='GET')
        def api_search(request, query):
            search_query = query.get('q', [''])[0]
            return self._search_posts(search_query)
        
        @self.web.route('/api/stats', method='GET')
        def api_stats(request, query):
            return self._get_stats()
        
        @self.web.route('/post/create', method='POST')
        def create_post_form(request, data):
            """フォームからの記事作成"""
            return self._handle_post_form(data)
    

@self.関数（）内の記号の時　def関数（その後につつく情報をよこせ）＃(request, data)

パターン使う場面
return self._render_〇〇()  IDや検索ワード不要  / /posts /editor
post_id = ... が必要    どの記事か特定が必要    /post/<id> /editor/<id>
search_query = ... が必要    検索ワードの取り出しが必要     /search?q=〇〇

In [ ]:
    def _create_post(self, post_data):
        """記事を作成してデータベースに保存（技術6）
        
        Cache-Aside パターンでキャッシュを無効化。
        """
        counter = self.db.get('post_counter')
        post_id = f"post_{counter}"
        counter += 1
        self.db.set('post_counter', counter)
        
        if isinstance(post_data, dict):
            post = post_data.copy()     ##辞書に変形。（ここは辞書ならコピペ）
        else:
            post = {}
            for key, value in post_data.items():  #じゃないとキーとバリューだし
                post[key] = value[0] if isinstance(value, list) else value
                ##値がリストなら最初を（value[0]~list).それ以外はそのまま入れる（Value)
        
        if 'tags' in post and isinstance(post['tags'], str):
            ## 例: tags = 'Python,Database,Tutorial' という状態だったら
            post['tags'] = [t.strip() for t in post['tags'].split(',') if t.strip()]
            # 「カンマで分割してリストに変換する」変数t.に対して処理する。strip() = 前後の空白を取り除く関数
        
        if 'date' not in post:      #postにdataがないならセットする
            post['date'] = datetime.now().strftime('%Y-%m-%d')
        if 'id' not in post:
            post['id'] = post_id    #さっき作った post_id をセットする」
        
        self.db.set(post_id, json.dumps(post))  #postをJSON文字列に変換してDBに保存する
        self.cache.delete('all_posts')
        
        return {'success': True, 'id': post_id, 'post': post} ## 「成功・記事ID・記事データを辞書にして返す」

Self.dbを使ったらカウント1つ追加してセットし直している。
初めのIfでデータの統一を計る。          t.strip()は前後の空白をなくす
tags = 'Python,Database,Tutorial'を ['Python', 'Database', 'Tutorial']する
また最後のは「ユーザーの入力ミス（余分なカンマや空白）で生まれる空っぽの要素を除外するため」

In [ ]:
def _get_post(self, post_id):   ##selfとpost_idで引っ張るDB
        """記事を取得（Cache-Aside パターン: キャッシュ → DB）"""
        cached = self.cache.get(f"post:{post_id}")  #本来はselfなし
        if cached:
                return json.loads(cached)# cached が None でなければここで終了
        try:
            post_json = self.db.get(post_id)
            post = json.loads(post_json)
            self.cache.set(f"post:{post_id}", post_json)#（）のPostで取り出したのを変換
            return post
        except KeyError:
            return None
    
    def _get_all_posts(self):
        """全記事を取得（キャッシュ付き）"""
        cached = self.cache.get('all_posts')
        if cached:
            return json.loads(cached)
        
        posts = []
        for key, value in self.db.all_items():
            if key.startswith('post_') and key != 'post_counter':
                #「キーが post_ で始まる かつ post_counter じゃない場合」
                # post_0, post_1 など記事だけを対象にする
                try:
                    post = json.loads(value) ###辞書変形
                    posts.append(post)   #リストに追加
                except (json.JSONDecodeError, TypeError): ##（変換できないのはスキップ
                    pass
        
        posts.sort(key=lambda x: x.get('date', ''), reverse=True)#sort並び替え、date基準
        self.cache.set('all_posts', json.dumps(posts))
        
        return posts
    
    

npost_id = 'post_0'
# f なし
"post:{post_id}"   → 'post:{post_id}'  # そのまま文字として表示 変数として認識されない
# f あり
f"post:{post_id}"  → 'post:post_0' 

loadsは辞書変換（post = json.loads(post_json)）
key=lambda x: x.get('date', '')
# 「各記事の date を並び替えの基準にする」辞書 x から date の値を取り出す、なければ空文字を返
 次回のためにキャッシュ保存